In [ ]:
%run _bootstrap_dev.ipynb

# Lazy Portfolio Analyst
Flusso: frequency selection → posizionamento rispetto alla frontiera
efficiente (diagnostico) → backtest PTF → validazione statistica.

La frontiera efficiente è uno strumento diagnostico: i suoi pesi
"ottimali" sono stimati in-sample sull'intero storico e non vanno
adottati come allocazione — sono affetti da overfitting strutturale
(si veda §5).

## §1 — Configurazione

In [ ]:
# Portafoglio da analizzare (definiti in l_portfolios.py)
# my_portfolio       = greta_base_spy_portfolio_etf_ita
# my_portfolio       = greta_alt_emdiv_test
my_portfolio       = lazy_greta_base_spy
# my_portfolio_title = "Gretchen S&P 500 Core-Engine - Fiduciaria"
# my_portfolio_title = "Gretchen S&P 500 Core-Engine - Alternative Test"
my_portfolio_title = "Gretchen S&P 500 Base SPY"

# Parametri analisi
start_date = '2016-01-01'   # inizio backtest
end_date   = None           # None = oggi
benchmark  = 'SPY'
init_cash  = 100_000
fees       = 0.001
years      = 10             # finestra frontiera efficiente (anni)

## §2 — Frequency Selection
Testa W/M/Q/Y/None e seleziona la frequenza con Sharpe massimo.
Cambia `freq_selection_metric` per usare 'cagr' | 'total_return' | 'max_dd'.

In [ ]:
freqs = ['W', 'M', 'Q', 'Y', None]
_rows = []
for _freq in freqs:
    _pf = run_bh_backtest(my_portfolio, start_date, end_date,
                          init_cash, fees, _freq)
    _eq = _pf.value()
    if isinstance(_eq, pd.DataFrame): _eq = _eq.iloc[:,0]
    _eq = _eq.dropna()
    _yrs = len(_eq) / 252
    _cagr = (_eq.iloc[-1]/_eq.iloc[0])**(1/_yrs)-1 if _yrs > 0 else np.nan
    _rows.append({
        'Freq'        : _freq if _freq is not None else 'BH',
        'Sharpe'      : float(_pf.sharpe_ratio()),
        'CAGR%'       : round(_cagr * 100, 2),
        'TotalReturn%': round(float(_pf.total_return()) * 100, 2),
        'MaxDD%'      : round(abs(float(_pf.max_drawdown())) * 100, 2),
    })
freq_df = pd.DataFrame(_rows)
my_display(freq_df, title="Confronto frequenze di ribilanciamento")

# Selezione automatica: max Sharpe (modifica idxmax → idxmin per MaxDD%)
freq_selection_metric = 'Sharpe'
best_label = freq_df.loc[freq_df[freq_selection_metric].idxmax(), 'Freq']
best_freq  = None if best_label == 'BH' else best_label
print(f"\n✅ Frequenza ottimale ({freq_selection_metric}): {best_label}")

## §3 — Posizionamento rispetto alla frontiera efficiente (diagnostico)

Mostra dove si colloca il PTF proposto rispetto allo spazio
rischio/rendimento stimato storicamente. Non è un suggerimento di
allocazione: i punti della frontiera (Min Vol, Max Sharpe, Max Return)
sono ottimizzati in-sample e quindi inevitabilmente overfit — si veda
la nota in §5.

**Legenda colonne:**
- `Return` / `Volatility` / `Sharpe` — valori teorici stimati da
  media storica e covarianza (PyPortfolioOpt, in-sample)
- `Real Return` / `Real Volatility` / `Real Sharpe` — valori
  effettivi misurati sul backtest reale nel periodo selezionato

In [ ]:
my_tickers = list(my_portfolio.keys())
my_weights = list(my_portfolio.values())

fig_frontier, df_special = efficient_frontier_pypfopt(
    tickers=my_tickers,
    years=years,
    my_weights=my_weights,
    n_points=80,
    weight_bounds=(0, 1),
    show_plot=True,
    interactive=True,
    print_weights=True,
)
# fig_frontier.show()

# Pesi ottimali Max Sharpe
_metric_cols = {'Return', 'Volatility', 'Sharpe',
                'Real Return', 'Real Volatility', 'Real Sharpe'}
_weight_cols = [c for c in df_special.columns if c not in _metric_cols]
optimal_weights = df_special.loc['Max Sharpe', _weight_cols].to_dict()

print("\nPosizionamento PTF proposto vs frontiera (diagnostico):")
print(f"  My Portfolio  — Sharpe: {df_special.loc['My Portfolio','Sharpe']:.2f}"
      f"  Return: {df_special.loc['My Portfolio','Return']:.2f}%"
      f"  Vol: {df_special.loc['My Portfolio','Volatility']:.2f}%")
print(f"  Max Sharpe pt — Sharpe: {df_special.loc['Max Sharpe','Sharpe']:.2f}"
      f"  Return: {df_special.loc['Max Sharpe','Return']:.2f}%"
      f"  Vol: {df_special.loc['Max Sharpe','Volatility']:.2f}%")
print("  ⚠️  Il punto Max Sharpe è in-sample — vedi §5 prima di interpretarlo")
# optimal_weights resta calcolato per riferimento (usato in §5 nota), non come allocazione consigliata

## §4 — Backtest PTF proposto
Analisi completa con la frequenza ottimale selezionata in §2.

In [ ]:
benchmark_data = download_data(benchmark, start_date, end_date)

pf_proposed = run_bh_backtest(my_portfolio, start_date, end_date,
                               init_cash, fees, best_freq)

portfolio_title = f"{my_portfolio_title} - Frequenza {best_freq}"

_ = generate_lazy_portfolio_performance(
    pf=pf_proposed,
    portfolio_title=portfolio_title,
    benchmark=benchmark,
    benchmark_data=benchmark_data,
    show_report=True,
    show_plots=True,
    alpha_analysis=True,
)

## §5 — Perché non adottiamo i pesi "ottimali" della frontiera

Il punto Max Sharpe calcolato in §3 massimizza lo Sharpe ratio
**sullo stesso campione storico** usato per stimarlo (rendimenti
attesi `mu` e covarianza `S` di PyPortfolioOpt). È l'equivalente di
un WFO con un solo fold in-sample e zero fold out-of-sample —
esattamente la trappola che il framework R-portfolio e K-strategy è
costruito per evitare.

In pratica: `mu` (rendimento atteso storico) è il parametro più
instabile e meno affidabile della teoria di Markowitz. I pesi
ottimali calcolati su un campione spesso non reggono fuori campione,
e talvolta perdono anche contro allocazioni banali (equal-weight,
60/40).

**Uso corretto della frontiera**: diagnostico (§3), non prescrittivo.
Se il PTF proposto è molto sotto la frontiera, è un segnale per
rivedere la diversificazione — non per copiare i pesi del punto
Max Sharpe.

La validazione statistica in §6 si applica quindi al **PTF reale
dell'utente** (`pf_proposed`), non ai pesi teorici della frontiera.

## §6 — Validazione statistica

Applicata al PTF reale dell'utente (pf_proposed), non ai pesi teorici
della frontiera. Stability pesi · MC Block A (confidence intervals) ·
MC Block B (skill ribilanciamento) · DSR · Decisione finale.

### §6.1 — Stability test pesi ottimali della frontiera (solo come misura di rumore)

Nota: questo test misura quanto è rumorosa la frontiera stessa
(quanto cambiano i suoi pesi Max Sharpe tra sotto-periodi) — NON
valida i pesi del mio PTF. Un CV alto qui conferma che il punto
Max Sharpe di §3 non va preso come riferimento di allocazione.

In [ ]:
stability = lazy_stability_weights(
    tickers=my_tickers,
    years=years,
    weight_bounds=(0, 1),
    n_splits=5,
    verbose=True,
)
print(f"\nRumorosità frontiera: {'bassa ✅' if stability['stable'] else 'alta ⚠️ (conferma: non adottare Max Sharpe come pesi)'}"
      f" (CV medio={stability['cv_mean']:.3f}, soglia={stability['cv_threshold']})")

In [ ]:
# print(stability['df_weights'])

### §6.2 — Monte Carlo Block A: Confidence Intervals sul PTF reale

Stima la distribuzione delle metriche (CAGR, Sharpe, MaxDD) di
pf_proposed via bootstrap.
- **A1 IID**: baseline, campiona ritorni indipendenti
- **A2 Block**: metodo principale, preserva autocorrelazione

In [ ]:
_rng = np.random.default_rng(42)
_n_sim = 1000
_block_size = 20  # ~1 mese trading

print("=== A1 — IID Bootstrap ===")
mc_a1 = mc_run_iid_bootstrap(pf_proposed, n_simulations=_n_sim, rng=_rng)

print("\n=== A2 — Block Bootstrap ===")
_rng2 = np.random.default_rng(42)
mc_a2 = mc_run_block_bootstrap(pf_proposed, block_size=_block_size,
                                  n_simulations=_n_sim, rng=_rng2)

print("\n--- Confidence Intervals (A2 Block Bootstrap) — PTF reale ---")
for metric in ['CAGR', 'Sharpe', 'MaxDD']:
    p5  = mc_a2['percentiles']['p5'][metric]
    p50 = mc_a2['percentiles']['p50'][metric]
    p95 = mc_a2['percentiles']['p95'][metric]
    act = mc_a2['actual_metrics'][metric]
    print(f"  {metric:8s}: actual={act:.3f}  p5={p5:.3f}  p50={p50:.3f}  p95={p95:.3f}")

### §6.3 — Monte Carlo Block B: Skill del ribilanciamento

Testa se la frequenza scelta in §2 per il MIO PTF aggiunge valore
vs date di ribilanciamento randomizzate (jitter ±30 giorni).
p-value < 0.05 → skill significativa.

In [ ]:
mc_b = lazy_mc_block_b_rebalancing(
    portfolio=my_portfolio,
    start_date=start_date,
    end_date=end_date,
    best_freq=best_freq,
    n_simulations=500,
    jitter_days=30,
    init_cash=init_cash,
    fees=fees,
    verbose=True,
)

### §6.4 — DSR: Deflated Sharpe Ratio del PTF reale

Valida la significatività statistica dello Sharpe ratio di pf_proposed.
DSR > 0 → Sharpe statisticamente significativo dopo correzione
per multiple testing.

In [ ]:
_sr  = float(pf_proposed.sharpe_ratio())
_T   = int(pf_proposed.value().dropna().__len__())
_dsr = ofc_compute_dsr(sr_hat=_sr, n_trials=1, T=_T)
print(f"Sharpe ratio  : {_sr:.3f}")
print(f"T (obs)       : {_T}")
print(f"DSR           : {_dsr:.3f}  {'✅ significativo' if _dsr > 0 else '⚠️ non significativo'}")

### §6.5 — Decisione finale

Valuta il PTF reale dell'utente (pesi e frequenza scelti in §1-§2),
non i pesi teorici della frontiera.

In [ ]:
cornice = max(65,len(portfolio_title)+10)

print("=" * cornice)
print(f"  LAZY PORTFOLIO — DECISIONE FINALE")
print(f"  {portfolio_title}")
print("=" * cornice)

_checks = {
    'MC A2 Sharpe p50>0'  : mc_a2['percentiles']['p50']['Sharpe'] > 0,
    'MC B skill rebalance': mc_b['skill'],
    'DSR > 0'             : _dsr > 0,
}
_passed = sum(_checks.values())
for label, ok in _checks.items():
    print(f"  {'✅' if ok else '❌'}  {label}")
print("-" * cornice)
print(f"  Criteri superati: {_passed}/{len(_checks)}")
print(f"  Verdetto: {'✅ PROMOSSO' if _passed >= 2 else '❌ RIGETTATO'}")
print("=" * cornice)